# Experiment 11 — Sensitization, tolerance and a dose-dependent host

**Question.** The model assumes that inflammation *lowers* the response threshold (sensitization, as in trained immunity). Periodontal tissues also show tolerance. What happens to the main predictions if the net effect of past inflammation is tolerizing, or if its sign depends on how intense the inflammation was?

**Memory target.** The memory variable relaxes towards $A^*(H)=aH-bH^2$ at rate $\varepsilon$, with $\theta=\theta_0e^{-A}$.
* $b=0,\ a>0$: sensitizing host (Exps 01–10).
* $b=0,\ a<0$: tolerizing host. Inflammation *raises* the threshold.
* $a>0,\ b>0$: dose-dependent host. Moderate inflammation trains ($A^*>0$), intense inflammation tolerizes ($A^*<0$ when $H>a/b$).

**Backbone.** We use the bistable backbone of Exps 01–03 ($n=q=4$, $K_M=0.5$, $\theta_0=0.6$, $r=1$). It is bistable for every $a\in[-0.5,0.5]$, so the same tests can be run for both signs. The comparator of Exps 06B–10 is not usable here, because its window exists only thanks to sensitization and vanishes for $a\le0$.

**Hosts.** Slow memory $\varepsilon=0.03$, and the instant host ($A=A^*(H)$), which has the same equilibria and no memory.

**Sections**
* **A** equilibrium structure for both signs
* **B** tipping window: naive vs treated sites, $a\in\{-0.5,-0.25,0,0.25,0.5\}$
* **C** relapse without challenge after therapy
* **D** periodic plaque forcing: conversion thresholds and repeated debridement
* **E** the dose-dependent host: naive, resolved gingivitis and treated disease
* **F** checks, summary and export

In [ ]:
# =====================================================================
# EXPERIMENT 11 — folder setup
# =====================================================================
import os
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path("/content/drive/MyDrive/Biosemiotic_Perio")
except ImportError:                      # local execution
    BASE_DIR = Path.cwd() / "Biosemiotic_Perio"
EXP_DIR = BASE_DIR / "Experiment_11_Sign_of_Adaptation"
EXP_DIR.mkdir(parents=True, exist_ok=True)
QUICK = os.environ.get("EXP_QUICK") == "1"     # set True for a fast smoke test with reduced grids
print(f"Experiment folder:\n{EXP_DIR}\nExists: {EXP_DIR.exists()}   QUICK={QUICK}")

In [ ]:
import sys, json, time, platform, shutil
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import scipy
from scipy.optimize import brentq

versions_text = f"""
Experiment: 11 Sign of adaptation
Created: {datetime.now(timezone.utc).isoformat()}

Python: {sys.version}
Platform: {platform.platform()}
NumPy: {np.__version__}
Pandas: {pd.__version__}
SciPy: {scipy.__version__}
Matplotlib: {matplotlib.__version__}
"""
with open(EXP_DIR / "software_versions.txt", "w") as f:
    f.write(versions_text)
print(versions_text)

# ---- figure style (same tokens as Experiment 07) ------------------------
INK, INK2, GRIDC, SURF = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
MEM_COL = {0.3: "#86b6ef", 0.1: "#5598e7", 0.03: "#2a78d6", 0.01: "#1c5cab", 0.003: "#0d366b"}
INSTANT_COL = "#eb6834"
plt.rcParams.update({
    "figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF,
    "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.grid": True, "grid.color": GRIDC, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 1.6,
    "font.size": 9, "axes.titlesize": 10, "axes.titleweight": "bold", "legend.frameon": False,
})
def save_fig(fig, name):
    fig.savefig(EXP_DIR / f"{name}.jpg", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# =====================================================================
# CORE MODEL FOR EXPERIMENT 11 — sign of adaptation
# =====================================================================
# State (H, M, A) with A = ln(theta0/theta). The memory target is a general
# function of inflammation,  A* = a*H - b*H**2:
#   b = 0, a > 0   sensitizing host (as in Exps 01-10)
#   b = 0, a < 0   tolerizing host (threshold rises with inflammation)
#   a > 0, b > 0   dose-dependent host: trained by moderate inflammation,
#                  tolerized by intense inflammation (target < 0 when H > a/b)
#   dH/dt = -H + F(M + u; theta0*exp(-A))
#   dM/dt = r*(-M + G(H))
#   dA/dt = eps*(A*(H) - A)
# mode 'adaptive' (slow memory) | 'instant' (A = A*(H): same equilibria, no memory).
# a, b and eps may be arrays (one value per element).
import numpy as np
from scipy.optimize import brentq

BACKBONE = dict(n=4.0, q=4.0, K=0.5, th0=0.6, r=1.0)    # bistable Exp 01-03 backbone
DT = 0.1


def hill(x, h, n):
    x = np.maximum(x, 0.0)
    xn = x ** n
    return xn / (xn + h ** n)


def target(H, a, b):
    return a * H - b * H ** 2


def rhs(H, M, A, u, eps, a, b, p, mode):
    At = target(H, a, b)
    A_eff = At if mode == 'instant' else A
    dH = -H + hill(M + u, p['th0'] * np.exp(-A_eff), p['n'])
    dM = p['r'] * (-M + hill(H, p['K'], p['q']))
    dA = eps * (At - A) if mode == 'adaptive' else 0.0 * A
    return dH, dM, dA


def run(y0, u_fn, n_steps, eps=0.0, a=1.0, b=0.0, mode='adaptive', p=BACKBONE, events=None, recorder=None, dt=DT):
    """Vectorized fixed-step RK4. u_fn(i, c) gives the forcing at step i, stage offset c in {0, .5, 1}.
    events(i, H, M, A) -> (H, M, A) is applied at the start of step i; recorder(i, H, M, A) after step i."""
    H, M, A = [np.array(v, float, copy=True) for v in np.broadcast_arrays(*y0)]
    eps, a, b = [np.broadcast_to(np.asarray(v, float), H.shape) for v in (eps, a, b)]
    f = lambda h, m, x, u: rhs(h, m, x, u, eps, a, b, p, mode)
    for i in range(n_steps):
        if events is not None:
            H, M, A = events(i, H, M, A)
        u0, uh, u1 = u_fn(i, 0.0), u_fn(i, 0.5), u_fn(i, 1.0)
        k1 = f(H, M, A, u0)
        k2 = f(H + .5*dt*k1[0], M + .5*dt*k1[1], A + .5*dt*k1[2], uh)
        k3 = f(H + .5*dt*k2[0], M + .5*dt*k2[1], A + .5*dt*k2[2], uh)
        k4 = f(H + dt*k3[0], M + dt*k3[1], A + dt*k3[2], u1)
        H = H + dt/6 * (k1[0] + 2*k2[0] + 2*k3[0] + k4[0])
        M = M + dt/6 * (k1[1] + 2*k2[1] + 2*k3[1] + k4[1])
        A = A + dt/6 * (k1[2] + 2*k2[2] + 2*k3[2] + k4[2])
        if recorder is not None:
            recorder(i, H, M, A)
    if mode == 'instant':
        A = target(H, a, b)
    return H, M, A


# ---------------------------------------------------------------- equilibria
X_GRID = np.linspace(-40, 14, 108001)
H_GRID = 1 / (1 + np.exp(-X_GRID))


def u_branch(H, a, b, p=BACKBONE):
    th = p['th0'] * np.exp(-target(H, a, b))
    return th * (H / (1 - H)) ** (1 / p['n']) - hill(H, p['K'], p['q'])


def folds(a, b, p=BACKBONE):
    """(H, u) of every fold of the equilibrium branch, sorted by H."""
    u = u_branch(H_GRID, a, b, p); d = np.gradient(u, X_GRID)
    idx = np.where(np.sign(d[:-1]) != np.sign(d[1:]))[0]
    out = []
    for i in idx:
        xf = brentq(lambda z: np.interp(z, X_GRID, d), X_GRID[i], X_GRID[i + 1])
        Hf = 1 / (1 + np.exp(-xf)); out.append((Hf, float(u_branch(Hf, a, b, p))))
    return out


def window(a, b, p=BACKBONE):
    """(u_down, u_up) of the bistable window, or (nan, nan) if the branch is monotone."""
    us = sorted(f[1] for f in folds(a, b, p))
    return (us[0], us[-1]) if len(us) >= 2 else (np.nan, np.nan)


def _roots(u, a, b, p):
    f = u_branch(H_GRID, a, b, p) - u
    idx = np.where(np.sign(f[:-1]) != np.sign(f[1:]))[0]
    return [brentq(lambda h: u_branch(h, a, b, p) - u, H_GRID[i], H_GRID[i + 1], xtol=1e-14) for i in idx]


def low_equilibrium(u, a, b, p=BACKBONE):
    H = _roots(u, a, b, p)[0]
    return H, hill(H, p['K'], p['q']), target(H, a, b)


def high_equilibrium(u, a, b, p=BACKBONE):
    H = _roots(u, a, b, p)[-1]
    return H, hill(H, p['K'], p['q']), target(H, a, b)

In [ ]:
# =====================================================================
# PARAMETERS
# =====================================================================
ALPHAS = np.array([-0.5, -0.25, 0.0, 0.25, 0.5])
EPS = 0.03
UB = 0.25                 # baseline plaque: inside the bistable window for every a in ALPHAS
U_PRE = 0.30              # plaque at which treated sites were diseased (Section C)
D_CH, SETTLE, SETTLE_I = 2 / EPS, 10 / EPS, 200.0
U_CH_MAX, ITERS = 4.0, 16
REC_X = np.array([0, 0.25, 0.5, 1, 2, 4], float)        # eps * T_rec
T, K_REG = 60.0, 0.05                                   # periodic forcing (Section D)
PEAKS = np.round(np.arange(0.20, 0.901, 0.005), 3)
N_CYC, N_VISITS = 40, 60
BIPHASIC = (1.0, 1.4)     # a, b: trained below H = a/b = 0.71, tolerized above
U_GING = 0.33             # sub-threshold episode for the dose-dependent host (below its upward fold)
if QUICK:
    ALPHAS = np.array([-0.25, 0.0, 0.5]); REC_X = np.array([0, 1], float); ITERS = 8
    PEAKS = np.round(np.arange(0.25, 0.701, 0.05), 3); N_CYC, N_VISITS = 12, 10

host_col = lambda a: {-0.5: "#c45a1d", -0.25: "#eb6834", 0.0: "#8a8984", 0.25: "#5598e7", 0.5: "#1c5cab"}.get(round(float(a), 2), INK)

def converts(y0, uch, eps, a, b, mode, d_ch, settle, ub):
    """Challenge of amplitude uch (array) for d_ch, then return to baseline ub for settle; True if H > 0.5 at the end."""
    n_ch, n_set = int(round(d_ch / DT)), int(round(settle / DT))
    ubv = np.broadcast_to(np.asarray(ub, float), np.shape(uch))
    H, M, A = run(y0, lambda i, c: uch if i < n_ch else ubv, n_ch + n_set, eps, a, b, mode)
    return H > 0.5

def critical_challenge(y0, eps, a, b, mode, d_ch, settle, ub, hi=U_CH_MAX, iters=ITERS):
    """Smallest converting challenge, by bisection (conversion is monotone in amplitude). nan: no conversion up to hi."""
    n = np.size(a); lo = np.full(n, float(np.max(ub))); hi = np.full(n, hi)
    ok = converts(y0, hi, eps, a, b, mode, d_ch, settle, ub)
    for _ in range(iters):
        mid = (lo + hi) / 2; c = converts(y0, mid, eps, a, b, mode, d_ch, settle, ub)
        hi = np.where(c, mid, hi); lo = np.where(c, lo, mid)
    return np.where(ok, hi, np.nan)
print("parameters set; QUICK =", QUICK)

## A. Equilibrium structure for both signs of adaptation

In [ ]:
rowsA = []
for a in np.round(np.arange(-0.5, 0.501, 0.05), 3):
    ud, uu = window(a, 0.0)
    rowsA.append(dict(a=a, b=0.0, u_down=ud, u_up=uu, width=uu - ud))
ud, uu = window(*BIPHASIC); rowsA.append(dict(a=BIPHASIC[0], b=BIPHASIC[1], u_down=ud, u_up=uu, width=uu - ud))
dfA = pd.DataFrame(rowsA); dfA.to_csv(EXP_DIR / "equilibrium_windows.csv", index=False)
display(dfA.round(4))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6), constrained_layout=True)
g = dfA[dfA.b == 0]
ax[0].fill_between(g.a, g.u_down, g.u_up, color="#cde2fb", lw=0)
ax[0].plot(g.a, g.u_up, color=INK, label="upward fold"); ax[0].plot(g.a, g.u_down, color=INK, ls="--", label="downward fold")
ax[0].axhline(UB, color=INK2, lw=0.8, ls=":"); ax[0].axhline(0, color=INK2, lw=0.6)
ax[0].axvline(0, color=INK2, lw=0.6)
ax[0].set(xlabel="adaptation a  (< 0 tolerizing, > 0 sensitizing)", ylabel="forcing u", title="Bistable window")
ax[0].legend(fontsize=8)
Hs = np.linspace(1e-4, 1 - 1e-4, 4000)
for a in ALPHAS:
    u = u_branch(Hs, a, 0.0); st = np.gradient(u, Hs) > 0
    ax[1].plot(np.where(st, u, np.nan), Hs, color=host_col(a), label=f"a = {a:g}")
    ax[1].plot(np.where(~st, u, np.nan), Hs, color=host_col(a), lw=0.7, ls="--")
u = u_branch(Hs, *BIPHASIC); st = np.gradient(u, Hs) > 0
ax[1].plot(np.where(st, u, np.nan), Hs, color="#1baf7a", label=f"dose-dependent {BIPHASIC}")
ax[1].plot(np.where(~st, u, np.nan), Hs, color="#1baf7a", lw=0.7, ls="--")
ax[1].axvline(UB, color=INK2, lw=0.8, ls=":")
ax[1].set(xlim=(-0.35, 0.7), xlabel="forcing u", ylabel="H*", title="Equilibrium branches"); ax[1].legend(fontsize=7)
save_fig(fig, "fig11_01_equilibria")

## B. Tipping window: naive vs treated sites

Sites sit at baseline $u_b=0.25$. *Naive* sites start on the low equilibrium. *Treated* sites start with $H$ and $M$ at their healthy values but with the memory $A$ they had on the high equilibrium at $u_b$: negative for a tolerizing host, positive for a sensitizing one. After recovery $T_{\text{rec}}$ at $u_b$, a challenge of duration $2/\varepsilon$ is applied. The critical challenge is found by bisection.

In [ ]:
t0 = time.time()
low = {a: low_equilibrium(UB, a, 0.0) for a in ALPHAS}; high = {a: high_equilibrium(UB, a, 0.0) for a in ALPHAS}
aa, xx = np.meshgrid(ALPHAS, REC_X, indexing='ij'); aa, xx = aa.ravel(), xx.ravel()
H0 = np.array([low[a][0] for a in aa]); M0 = np.array([low[a][1] for a in aa])
A_treat = np.array([high[a][2] for a in aa]); A_naive = np.array([low[a][2] for a in aa])
# recovery of treated sites at baseline for eps*T_rec (vectorized: elements stop changing once their time is reached)
n_rec = np.round(xx / EPS / DT).astype(int)
state = [H0.copy(), M0.copy(), A_treat.copy()]
H, M, A = H0.copy(), M0.copy(), A_treat.copy()
def rec_fn(i, h, m, x):
    done = (i + 1) == n_rec
    for s, v in zip(state, (h, m, x)):
        s[done] = v[done]
if n_rec.max() > 0:
    run((H, M, A), lambda i, c: np.full(aa.size, UB), int(n_rec.max()), EPS, aa, 0.0, 'adaptive', recorder=rec_fn)
crit_treated = critical_challenge(tuple(state), EPS, aa, 0.0, 'adaptive', D_CH, SETTLE, UB)
crit_naive = critical_challenge((H0[::len(REC_X)], M0[::len(REC_X)], A_naive[::len(REC_X)]), EPS, ALPHAS, 0.0, 'adaptive', D_CH, SETTLE, UB)
crit_inst = critical_challenge((H0[::len(REC_X)], M0[::len(REC_X)], A_naive[::len(REC_X)]), 0.0, ALPHAS, 0.0, 'instant', D_CH, SETTLE_I, UB)
naive_map = dict(zip(ALPHAS, crit_naive)); inst_map = dict(zip(ALPHAS, crit_inst))
dfB = pd.DataFrame(dict(a=aa, eps_T_rec=xx, A_after_therapy=A_treat, critical_treated=crit_treated,
                        critical_naive=[naive_map[a] for a in aa], critical_instant=[inst_map[a] for a in aa]))
dfB['ratio_treated_to_naive'] = dfB.critical_treated / dfB.critical_naive
c0 = dfB[dfB.eps_T_rec == 0].set_index('a').critical_treated
dfB['V'] = (dfB.critical_naive - dfB.critical_treated) / (dfB.critical_naive - dfB.a.map(c0))
dfB.to_csv(EXP_DIR / "tipping_window_by_sign.csv", index=False)
print(f"done in {time.time() - t0:.0f}s  (nan = no conversion up to u_ch = {U_CH_MAX})")
display(dfB[dfB.eps_T_rec == 0][['a', 'A_after_therapy', 'critical_naive', 'critical_treated', 'ratio_treated_to_naive', 'critical_instant']].round(3))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6), constrained_layout=True)
g = dfB[dfB.eps_T_rec == 0]
ax[0].axhline(1, color=INK2, lw=0.8)
ax[0].plot(g.a, g.ratio_treated_to_naive, "o-", color=INK)
ax[0].set(xlabel="adaptation a  (< 0 tolerizing, > 0 sensitizing)", ylabel="critical challenge, treated / naive",
          title="Treated sites right after therapy")
ax[0].text(0.03, 0.06, "above 1: therapy leaves the site more resistant\nbelow 1: more fragile", transform=ax[0].transAxes, fontsize=8, color=INK2)
for a in ALPHAS:
    if a == 0: continue
    gg = dfB[dfB.a == a].sort_values('eps_T_rec')
    ax[1].plot(gg.eps_T_rec, gg.ratio_treated_to_naive, "o-", color=host_col(a), label=f"a = {a:g}")
ax[1].axhline(1, color=INK2, lw=0.8)
ax[1].set(xlabel="recovery time eps*T_rec (memory units)", ylabel="critical challenge, treated / naive", title="Window closes as memory recovers")
ax[1].legend(fontsize=8)
save_fig(fig, "fig11_02_tipping_window_by_sign")

## C. Relapse without challenge after therapy

Treated sites carry the memory they had when diseased at $u=0.30$ (inside every window), with $H$ and $M$ restored to the healthy values at the new baseline $u_b$. A site relapses if it ends with $H>0.5$ after $10/\varepsilon$ at $u_b$, with no challenge.

In [ ]:
t0 = time.time(); rowsC = []
for a in ALPHAS:
    ud, uu = window(a, 0.0); A_pre = high_equilibrium(U_PRE, a, 0.0)[2]
    ubs = np.round(np.arange(0.005, uu - 0.004, 0.005), 3)
    lows = [low_equilibrium(u, a, 0.0) for u in ubs]
    H0 = np.array([l[0] for l in lows]); M0 = np.array([l[1] for l in lows])
    for mode, eps, A0 in [('adaptive', EPS, np.full(ubs.size, A_pre)), ('instant', 0.0, np.array([l[2] for l in lows]))]:
        H, M, A = run((H0, M0, A0), lambda i, c: ubs, int((SETTLE if mode == 'adaptive' else SETTLE_I) / DT), eps, a, 0.0, mode)
        rel = H > 0.5
        ceiling = ubs[rel].min() if rel.any() else np.nan
        rowsC.append(dict(a=a, host='instant' if mode == 'instant' else f'eps={eps}', A_memory=A_pre if mode == 'adaptive' else 0.0,
                          relapse_ceiling=ceiling, static_fold=uu, ceiling_over_fold=ceiling / uu))
dfC = pd.DataFrame(rowsC); dfC.to_csv(EXP_DIR / "relapse_ceiling_by_sign.csv", index=False)
print(f"done in {time.time() - t0:.0f}s  (nan = no relapse below the static fold)"); display(dfC.round(3))

## D. Periodic plaque forcing: conversion thresholds and repeated debridement

As in Exp 07: plaque regrows as $u_{max}(1-e^{-k\tau})$, $k=0.05$, and is removed every $T=60$; naive sites. Conversion: entrapped ($H>0.5$ throughout the cycle) in each of the last three cycles. Then, at a peak above both thresholds, debridement ($M\to0$) at every visit, recording the fraction of each interval with $H<0.5$.

In [ ]:
t0 = time.time(); Ns = int(round(T / DT)); f_pk = 1 - np.exp(-K_REG * T)
def periodic_u(umax):
    return lambda i, c: umax * (1 - np.exp(-K_REG * (((i % Ns) + c) * DT)))
rowsD = []
for mode, eps in [('adaptive', EPS), ('instant', 0.0)]:
    aa, pp = np.meshgrid(ALPHAS, PEAKS, indexing='ij'); aa, pp = aa.ravel(), pp.ravel()
    Hmin = np.full((aa.size, N_CYC), np.nan); cur = {}
    def rec(i, H, M, A):
        if i % Ns == 0: cur['m'] = H.copy()
        cur['m'] = np.minimum(cur['m'], H)
        if (i + 1) % Ns == 0: Hmin[:, (i + 1) // Ns - 1] = cur['m']
    run((np.zeros(aa.size), 0.0, 0.0), periodic_u(pp / f_pk), N_CYC * Ns, eps, aa, 0.0, mode, recorder=rec)
    conv = (Hmin[:, -3:] > 0.5).all(axis=1)
    for a in ALPHAS:
        m = (aa == a) & conv
        rowsD.append(dict(a=a, host='instant' if mode == 'instant' else f'eps={eps}',
                          threshold_peak=pp[m].min() if m.any() else np.nan, static_fold=window(a, 0.0)[1]))
dfD = pd.DataFrame(rowsD); dfD['threshold_over_fold'] = dfD.threshold_peak / dfD.static_fold
dfD.to_csv(EXP_DIR / "periodic_thresholds_by_sign.csv", index=False)
print(f"thresholds done in {time.time() - t0:.0f}s"); display(dfD.round(3))

# repeated debridement at a peak above both thresholds
t0 = time.time(); rowsD2 = []
for a in [x for x in ALPHAS if x != 0]:
    thr = dfD[dfD.a == a].threshold_peak.max()
    pk = float(np.round(thr + 0.05, 3)) if np.isfinite(thr) else 0.6
    for mode, eps in [('adaptive', EPS), ('instant', 0.0)]:
        free = np.zeros(N_VISITS); cnt = {'t': 0.0}
        def ev(i, H, M, A):
            if i % Ns == 0: M = np.zeros_like(M)
            return H, M, A
        def rec(i, H, M, A):
            c = i // Ns; free[c] += DT * float(H[0] < 0.5)
        run((np.zeros(1), 0.0, 0.0), periodic_u(np.array([pk / f_pk])), N_VISITS * Ns, eps, np.array([a]), 0.0, mode, events=ev, recorder=rec)
        for v in range(N_VISITS):
            rowsD2.append(dict(a=a, host='instant' if mode == 'instant' else f'eps={eps}', peak=pk, visit=v, relapse_free_fraction=free[v] / T))
dfD2 = pd.DataFrame(rowsD2); dfD2.to_csv(EXP_DIR / "debridement_by_sign.csv", index=False)
print(f"debridement done in {time.time() - t0:.0f}s")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6), constrained_layout=True)
for host, ls in [('instant', '--'), (f'eps={EPS}', '-')]:
    g = dfD[dfD.host == host]
    ax[0].plot(g.a, g.threshold_over_fold, "o" + ls, color=INK, label=host)
ax[0].axhline(1, color=INK2, lw=0.8)
ax[0].set(xlabel="adaptation a", ylabel="conversion threshold / static fold", title=f"Periodic forcing, T = {T:g}"); ax[0].legend(fontsize=8)
for (a, host), g in dfD2.groupby(['a', 'host']):
    ax[1].plot(g.visit, g.relapse_free_fraction, '-' if host != 'instant' else '--', color=host_col(a),
               label=f"a = {a:g}, {host}, peak {g.peak.iloc[0]:g}")
ax[1].set(xlabel="debridement visit", ylabel="relapse-free fraction", ylim=(0, 1.05), title="Debridement at every visit")
ax[1].legend(fontsize=6.5)
save_fig(fig, "fig11_03_periodic_by_sign")

## E. The dose-dependent host

Target $A^*(H)=aH-bH^2$ with $(a,b)=(1.0,1.4)$: memory is positive (trained) after moderate inflammation and negative (tolerized) after intense inflammation, crossing zero at $H=0.71$. Three histories at $u_b=0.25$: naive; resolved gingivitis ($u=0.33$, below the upward fold, for $5/\varepsilon$); treated disease (memory from the high equilibrium at $u_b$).

In [ ]:
t0 = time.time(); a_b, b_b = BIPHASIC
lo_b = low_equilibrium(UB, a_b, b_b); hi_b = high_equilibrium(UB, a_b, b_b)
# resolved gingivitis: exposure at U_GING for 5/eps from the naive state, then H and M back to healthy values
Hg, Mg, Ag = run((np.array([lo_b[0]]), lo_b[1], lo_b[2]), lambda i, c: np.array([U_GING]), int(5 / EPS / DT), EPS, a_b, b_b, 'adaptive')
hist = {'naive': lo_b[2], 'resolved gingivitis': float(Ag[0]), 'treated disease': hi_b[2]}
names = list(hist); xs = REC_X
nn, xx2 = np.meshgrid(np.arange(len(names)), xs, indexing='ij'); nn, xx2 = nn.ravel(), xx2.ravel()
A0 = np.array([hist[names[k]] for k in nn]); H0 = np.full(nn.size, lo_b[0]); M0 = np.full(nn.size, lo_b[1])
n_rec = np.round(xx2 / EPS / DT).astype(int); state = [H0.copy(), M0.copy(), A0.copy()]
def rec_fn(i, h, m, x):
    done = (i + 1) == n_rec
    for s, v in zip(state, (h, m, x)):
        s[done] = v[done]
if n_rec.max() > 0:
    run((H0, M0, A0), lambda i, c: np.full(nn.size, UB), int(n_rec.max()), EPS, a_b, b_b, 'adaptive', recorder=rec_fn)
cE = critical_challenge(tuple(state), EPS, np.full(nn.size, a_b), b_b, 'adaptive', D_CH, SETTLE, UB)
dfE = pd.DataFrame(dict(history=[names[k] for k in nn], eps_T_rec=xx2, A_start=A0, critical=cE))
cn = dfE[(dfE.history == 'naive')].critical.iloc[0]; dfE['ratio_to_naive'] = dfE.critical / cn
dfE.to_csv(EXP_DIR / "dose_dependent_host.csv", index=False)
print(f"done in {time.time() - t0:.0f}s; memory after each history:", {k: round(v, 3) for k, v in hist.items()})
display(dfE[dfE.eps_T_rec == 0].round(3))

fig, ax = plt.subplots(figsize=(5.2, 3.6), constrained_layout=True)
for h, col in zip(names, [INK2, "#1baf7a", "#6250d6"]):
    g = dfE[dfE.history == h]; ax.plot(g.eps_T_rec, g.ratio_to_naive, "o-", color=col, label=h)
ax.axhline(1, color=INK2, lw=0.8)
ax.set(xlabel="recovery time eps*T_rec", ylabel="critical challenge / naive", title="Dose-dependent host (trained, then tolerized)")
ax.legend(fontsize=8)
save_fig(fig, "fig11_04_dose_dependent")

## F. Checks, summary and export

In [ ]:
checks = []
g0 = dfB[(dfB.a == 0)]
checks.append(dict(check="a = 0: treated = naive at every recovery time", value=bool(np.allclose(g0.critical_treated, g0.critical_naive, equal_nan=True))))
r0 = dfB[dfB.eps_T_rec == 0].set_index('a').ratio_treated_to_naive
checks.append(dict(check="sensitizing hosts (a > 0): treated sites more fragile (ratio < 1)", value=bool((r0[r0.index > 0] < 1).all())))
checks.append(dict(check="tolerizing hosts (a < 0): treated sites more resistant (ratio > 1 or no conversion)",
                   value=bool(((r0[r0.index < 0] > 1) | r0[r0.index < 0].isna()).all())))
inst_vs_naive = dfB[dfB.eps_T_rec == 0][['a', 'critical_instant']]
checks.append(dict(check="instant host has no treated/naive distinction by construction", value=True))
dfChk = pd.DataFrame(checks); dfChk.to_csv(EXP_DIR / "checks.csv", index=False); display(dfChk)

summary = pd.concat([
    dfB[dfB.eps_T_rec == 0][['a', 'ratio_treated_to_naive']].assign(item=lambda d: "treated/naive critical challenge, a=" + d.a.astype(str)).rename(columns={'ratio_treated_to_naive': 'value'})[['item', 'value']],
    dfC.assign(item=lambda d: "relapse ceiling / static fold, " + d.host + ", a=" + d.a.astype(str)).rename(columns={'ceiling_over_fold': 'value'})[['item', 'value']],
    dfD.assign(item=lambda d: "periodic threshold / static fold, " + d.host + ", a=" + d.a.astype(str)).rename(columns={'threshold_over_fold': 'value'})[['item', 'value']],
    dfE[dfE.eps_T_rec == 0].assign(item=lambda d: "dose-dependent host, critical/naive, " + d.history).rename(columns={'ratio_to_naive': 'value'})[['item', 'value']],
])
summary.to_csv(EXP_DIR / "summary.csv", index=False); display(summary.round(3))
with open(EXP_DIR / "parameters.json", "w") as f:
    json.dump(dict(backbone=BACKBONE, alphas=ALPHAS.tolist(), eps=EPS, u_b=UB, u_pre=U_PRE, d_ch=D_CH, settle=SETTLE,
                   T=T, k=K_REG, biphasic=BIPHASIC, u_ging=U_GING, dt=DT), f, indent=2)

In [ ]:
readme = """Experiment 11 - Sign of adaptation (sensitizing, tolerizing, dose-dependent host)
Backbone n=q=4, K_M=0.5, theta0=0.6, r=1 (bistable for a in [-0.5, 0.5]). Memory target A*(H) = a*H - b*H^2.
Hosts: slow memory eps=0.03; instant (A = A*(H)).
Files
- equilibrium_windows.csv, fig11_01_equilibria             folds vs a; branches
- tipping_window_by_sign.csv, fig11_02                      critical challenge naive vs treated, by a and recovery time
- relapse_ceiling_by_sign.csv                               relapse without challenge after therapy
- periodic_thresholds_by_sign.csv, debridement_by_sign.csv, fig11_03   periodic forcing; debridement at every visit
- dose_dependent_host.csv, fig11_04                         trained-then-tolerized host: naive / gingivitis / treated
- checks.csv, summary.csv, parameters.json
"""
with open(EXP_DIR / "README.txt", "w") as f:
    f.write(readme)
archive = shutil.make_archive(str(BASE_DIR / "Experiment_11_Sign_of_Adaptation_EXPORT"), 'zip', EXP_DIR)
print("Saved:", archive)